## 📦 Kütüphaneleri İçe Aktar

In [1]:
import numpy as np
import pandas as pd
import warnings
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 📂 Veri Setini Yükle

In [2]:
# Kaggle dataset path
DATA_PATH = '/kaggle/input/bank-customer-churn-dataset/Bank Customer Churn Prediction.csv'

# Load data
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (10000, 12)


,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 🔧 Yeni Özellikler Oluştur

EDA'dan elde edilen içgörülere dayanarak, aşağıdaki özellikleri oluşturacağız:

1. **balance_to_salary_ratio** - Bakiye-maaş oranı
2. **tenure_age_ratio** - Müşteri olma süresi-yaş oranı
3. **credit_score_category** - Kredi skoru kategorileri (Düşük, Orta, Yüksek)
4. **age_group** - Yaş grupları (Genç, Orta Yaş, Yaşlı)
5. **high_value_customer** - Yüksek değerli müşteri flagleri
6. **inactive_high_balance** - Aktif olmayan + yüksek bakiye kombinasyonu

In [3]:
# Create a copy for feature engineering
df_fe = df.copy()

print("Original shape:", df_fe.shape)
print("Starting feature engineering...")

Original shape: (10000, 12)
Starting feature engineering...


In [4]:
# 1. Balance to Salary Ratio
df_fe['balance_to_salary_ratio'] = df_fe['balance'] / (df_fe['estimated_salary'] + 1)  # +1 to avoid division by zero

# 2. Tenure to Age Ratio
df_fe['tenure_age_ratio'] = df_fe['tenure'] / (df_fe['age'] + 1)

# 3. Credit Score Category
df_fe['credit_score_category'] = pd.cut(
    df_fe['credit_score'], 
    bins=[0, 600, 700, 850], 
    labels=['Low', 'Medium', 'High']
)

# 4. Age Group
df_fe['age_group'] = pd.cut(
    df_fe['age'], 
    bins=[0, 30, 50, 100], 
    labels=['Young', 'Middle-aged', 'Senior']
)

# 5. High Value Customer (high balance AND high salary)
balance_threshold = df_fe['balance'].quantile(0.75)
salary_threshold = df_fe['estimated_salary'].quantile(0.75)
df_fe['high_value_customer'] = (
    (df_fe['balance'] >= balance_threshold) & 
    (df_fe['estimated_salary'] >= salary_threshold)
).astype(int)

# 6. Inactive High Balance (not active member but high balance)
df_fe['inactive_high_balance'] = (
    (df_fe['active_member'] == 0) & 
    (df_fe['balance'] >= balance_threshold)
).astype(int)

print("\n✅ Feature Engineering Completed!")
print(f"New shape: {df_fe.shape}")
print(f"\nNew features created:")
new_features = ['balance_to_salary_ratio', 'tenure_age_ratio', 'credit_score_category', 
                'age_group', 'high_value_customer', 'inactive_high_balance']
for feat in new_features:
    print(f"  - {feat}")


✅ Feature Engineering Completed!
New shape: (10000, 18)

New features created:
  - balance_to_salary_ratio
  - tenure_age_ratio
  - credit_score_category
  - age_group
  - high_value_customer
  - inactive_high_balance


In [5]:
# Check new features
df_fe[new_features].head(10)

,balance_to_salary_ratio,tenure_age_ratio,credit_score_category,age_group,high_value_customer,inactive_high_balance
0,0.000000,0.046512,Medium,Middle-aged,0,0
1,0.744670,0.023810,Medium,Middle-aged,0,0
2,1.401362,0.186047,Low,Middle-aged,0,1
3,0.000000,0.025000,Medium,Middle-aged,0,0
4,1.587035,0.045455,High,Middle-aged,0,0
5,0.759599,0.177778,Medium,Middle-aged,0,0
6,0.000000,0.137255,High,Middle-aged,0,0
7,0.963961,0.133333,Low,Young,0,0
8,1.895493,0.088889,Low,Middle-aged,0,0
9,1.876621,0.071429,Medium,Young,0,0


## ✂️ Train-Test Ayrımı

In [6]:
# Original features
original_numerical = ['credit_score', 'age', 'tenure', 'balance', 'products_number', 'estimated_salary']
original_categorical = ['country', 'gender']
original_binary = ['credit_card', 'active_member']

# New features
new_numerical = ['balance_to_salary_ratio', 'tenure_age_ratio']
new_categorical = ['credit_score_category', 'age_group']
new_binary = ['high_value_customer', 'inactive_high_balance']

# Combined
all_numerical = original_numerical + new_numerical
all_categorical = original_categorical + new_categorical
all_binary = original_binary + new_binary

all_features = all_numerical + all_categorical + all_binary

print(f"Total features: {len(all_features)}")
print(f"  - Numerical: {len(all_numerical)}")
print(f"  - Categorical: {len(all_categorical)}")
print(f"  - Binary: {len(all_binary)}")

Total features: 16
  - Numerical: 8
  - Categorical: 4
  - Binary: 4


## 🔧 Ön İşleme Pipeline'ı Oluştur

In [7]:
# X and y
X = df_fe[all_features]
y = df_fe['churn']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTrain churn distribution:\n{y_train.value_counts()}")
print(f"\nTest churn distribution:\n{y_test.value_counts()}")

Train set: (8000, 16)
Test set: (2000, 16)

Train churn distribution:
churn
0    6370
1    1630
Name: count, dtype: int64

Test churn distribution:
churn
0    1593
1     407
Name: count, dtype: int64


## 🔄 Veriyi Ön İşle

In [8]:
# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), all_numerical),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), all_categorical)
    ],
    remainder='passthrough'  # Binary columns olduğu gibi
)

print("Preprocessing pipeline created!")
print(f"\nNumerical columns (RobustScaler): {all_numerical}")
print(f"Categorical columns (OneHotEncoder): {all_categorical}")
print(f"Binary columns (passthrough): {all_binary}")

Preprocessing pipeline created!

Numerical columns (RobustScaler): ['credit_score', 'age', 'tenure', 'balance', 'products_number', 'estimated_salary', 'balance_to_salary_ratio', 'tenure_age_ratio']
Categorical columns (OneHotEncoder): ['country', 'gender', 'credit_score_category', 'age_group']
Binary columns (passthrough): ['credit_card', 'active_member', 'high_value_customer', 'inactive_high_balance']


## 💾 Ön İşlenmiş Veriyi Kaydet

In [9]:
# Fit and transform
print("Transforming data...")
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"\n✅ Transformation completed!")
print(f"X_train_transformed shape: {X_train_transformed.shape}")
print(f"X_test_transformed shape: {X_test_transformed.shape}")

Transforming data...

✅ Transformation completed!
X_train_transformed shape: (8000, 19)
X_test_transformed shape: (2000, 19)


## 💾 Preprocessor'ı Kaydet

In [10]:
# Get feature names after OHE
feature_names = []

# Numerical features (no name change)
feature_names.extend(all_numerical)

# Categorical features (after OHE)
if hasattr(preprocessor.named_transformers_['cat'], 'get_feature_names_out'):
    cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(all_categorical)
    feature_names.extend(cat_features)
else:
    # Fallback for older sklearn versions
    feature_names.extend([f"{col}_encoded" for col in all_categorical])

# Binary features (no name change)
feature_names.extend(all_binary)

print(f"Total features after transformation: {len(feature_names)}")
print(f"\nFeature names:")
for i, name in enumerate(feature_names, 1):
    print(f"  {i}. {name}")

Total features after transformation: 19

Feature names:
  1. credit_score
  2. age
  3. tenure
  4. balance
  5. products_number
  6. estimated_salary
  7. balance_to_salary_ratio
  8. tenure_age_ratio
  9. country_Germany
  10. country_Spain
  11. gender_Male
  12. credit_score_category_Low
  13. credit_score_category_Medium
  14. age_group_Senior
  15. age_group_Young
  16. credit_card
  17. active_member
  18. high_value_customer
  19. inactive_high_balance


## 📊 Yeni Özellikleri Görselleştir

In [11]:
# Save to Kaggle working directory
output_dir = '/kaggle/working/'

# Save numpy arrays
np.save(output_dir + 'X_train_fe.npy', X_train_transformed)
np.save(output_dir + 'X_test_fe.npy', X_test_transformed)
np.save(output_dir + 'y_train.npy', y_train.values)
np.save(output_dir + 'y_test.npy', y_test.values)

# Save feature names
with open(output_dir + 'feature_names.txt', 'w') as f:
    for name in feature_names:
        f.write(f"{name}\n")

# Save preprocessor for later use
with open(output_dir + 'preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

print("✅ All files saved successfully!")
print(f"\nSaved files:")
print(f"  - X_train_fe.npy ({X_train_transformed.shape})")
print(f"  - X_test_fe.npy ({X_test_transformed.shape})")
print(f"  - y_train.npy ({y_train.shape})")
print(f"  - y_test.npy ({y_test.shape})")
print(f"  - feature_names.txt ({len(feature_names)} features)")
print(f"  - preprocessor.pkl")

✅ All files saved successfully!

Saved files:
  - X_train_fe.npy ((8000, 19))
  - X_test_fe.npy ((2000, 19))
  - y_train.npy ((8000,))
  - y_test.npy ((2000,))
  - feature_names.txt (19 features)
  - preprocessor.pkl


## 📝 Özellik Mühendisliği Özeti

### Oluşturulan Yeni Özellikler:

#### 1. Sayısal Özellikler:
- **balance_to_salary_ratio**: Bakiye-maaş oranı
  - Müşterinin finansal durumu hakkında içgörü sağlar
  - Yüksek oranlar = daha fazla tasarruf

- **tenure_age_ratio**: Müşteri olma süresi-yaş oranı
  - Müşteri sadakat yoğunluğunu gösterir
  - Yüksek oranlar = uzun vadeli sadakat

#### 2. Kategorik Özellikler:
- **credit_score_category**: Kredi skoru kategorisi
  - Düşük (300-580), Orta (580-670), İyi (670-740), Çok İyi (740-800), Mükemmel (800+)
  - Kredi riskini kategorize eder

- **age_group**: Yaş grupları
  - Genç (18-30), Orta Yaş (30-50), Yaşlı (50+)
  - Demografik segmentasyon

#### 3. Binary Özellikler:
- **high_value_customer**: Yüksek değerli müşteri
  - Yüksek bakiye VE yüksek maaş (her ikisi de 75. persentil üstü)
  - Değerli müşteriyi tanımlar

- **inactive_high_balance**: Aktif olmayan + yüksek bakiye
  - Aktif değil ANCAK yüksek bakiye var
  - Potansiyel kayıp riski göstergeleri

### Özet İstatistikler:
- **Toplam Özellikler**: 16 (10 orijinal + 6 yeni)
- **Sayısal**: 7 (5 orijinal + 2 yeni)
- **Kategorik**: 5 (3 orijinal + 2 yeni)
- **Binary**: 4 (2 orijinal + 2 yeni)

### Kaydedilen Dosyalar:
✅ **X_train_fe.npy** - Ön işlenmiş train verisi (8000, 19)
✅ **X_test_fe.npy** - Ön işlenmiş test verisi (2000, 19)
✅ **y_train.npy** - Train hedefleri (8000,)
✅ **y_test.npy** - Test hedefleri (2000,)
✅ **feature_names.txt** - Özellik isimleri (19 features)
✅ **preprocessor.pkl** - Ön işleme pipeline'ı

### Sonraki Adımlar:
1. Model optimizasyonu (XGBoost, LightGBM, CatBoost)
2. Hiperparametre ayarlama
3. SHAP değerleriyle özellik önemini analiz etme